In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Concise Implementation

In that section we wrote every part of softmax regression by
hand. In practice nobody does. We use the building blocks of a deep learning
framework instead, as that section did for linear regression:
a fully connected layer that holds the weights and the bias, and a
cross-entropy loss. If saving lines of code were all there is to it, this
section could be very short. The framework's loss, however, differs from ours
in one respect that matters: it takes the logits rather than the
probabilities. Recall that the softmax we wrote by hand returns `NaN` once a
logit grows large, and that the floor under its probabilities kept the loss
finite only by cutting off the gradient. How can the loss be computed so that
it stays finite, and keeps its gradient, for every value of the logits? The
answer, it turns out, is never to form the probabilities at all.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## The Model

As in that section, a fully connected layer of the framework
holds the weights $\mathbf{W}$ and the bias $\mathbf{b}$ and computes
$\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b}$. It has 10 outputs, one per
class. The model flattens each image into a vector of 784 pixel values
before the layer, as the scratch implementation did, and its `forward`
returns the logits. Note that we set the initial values ourselves: the
weights start from a normal distribution with mean 0 and standard deviation
0.01 and the biases at 0, again as in the scratch implementation. Each
framework's layer would otherwise draw its weights from a default
distribution of its own, and the four tabs would train different models.

Keras creates the weights of `Dense` at the first forward pass, once the
number of inputs is known, and draws them from the `kernel_initializer`.

In [ ]:
class SoftmaxRegression(d2l.Classifier):
    """The softmax regression model."""
    def __init__(self, num_outputs, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = tf.keras.models.Sequential()
        self.net.add(tf.keras.layers.Flatten())
        self.net.add(tf.keras.layers.Dense(
            num_outputs,
            kernel_initializer=tf.initializers.RandomNormal(stddev=0.01)))

    def forward(self, X):
        return self.net(X)

## A Stable Cross-Entropy Loss

### Overflow and Underflow

The scratch implementation computes the probabilities first and takes the
logarithm afterwards, and each step can fail in floating point. Single
precision stores numbers up to about $3.4 \times 10^{38}$, so $\exp(o)$
overflows to infinity once $o$ exceeds about 88, which is what turned the
softmax of $(1000, 0, 0)$ into `NaN` in that section. This
failure is the easy one. A common shift of all logits leaves the softmax
unchanged, so subtracting the largest logit, $o_{\max} = \max_k o_k$, from
each of them removes it, as we saw there. Unfortunately, the shift does
nothing for the lower end. For the logits $(0, -200, 0)$ the probability of
the second class is about $e^{-200}/2$, smaller than any positive
single-precision number, and so it is stored as 0; this happens to every
logit that lies more than about 104 below the largest one. The logarithm of
this 0 is $-\infty$, and backpropagation through it produces `NaN`. The
floor of the scratch loss hides the infinity and discards the gradient with
it. Both failures have the same cause: we formed a probability that the
loss never needed.

### The Log-Sum-Exp Form

What does the loss need? Only the logarithm of one probability, and the
logarithm of a quotient is a difference. Taking the logarithm of the softmax
on paper, before anything is computed, removes the division:

$$
\log \hat{y}_j = o_j - \log \sum_k \exp (o_k).
$$

This is the key step. For an example of true class $y$, the loss
$l = -\log \hat y_y$ becomes a function of the logits alone, with no
probability in between. As it stands the sum can still overflow, so we pull
the factor $\exp(o_{\max})$ out of it:

$$
l(y, \mathbf{o}) = \log \sum_{k} \exp(o_k) - o_y =
o_{\max} + \log \sum_{k} \exp(o_k - o_{\max}) - o_y.
$$

The two sides are equal, but only the right side should be handed to a
computer. All its exponents are at most 0, and one of them is equal to 0, so
the sum lies between 1 and $q$ and its logarithm between 0 and $\log q$.
Every term is an ordinary number, whatever the logits. For $(0, -200, 0)$
with the second class as the true one, the loss is
$\log 2 + 200 \approx 200.7$: large, as it should be for so wrong a
prediction, but finite. The first term, $\log \sum_k \exp(o_k)$, is the
*log-sum-exp* of the logits that that section
differentiated.

### The Framework Loss

The cross-entropy losses of the frameworks compute
the equation when they are given logits. They never
form the probabilities, so neither the exponential of a large number nor the
logarithm of 0 occurs. Their gradient is the gradient of this expression,
$\partial l / \partial o_j = \mathrm{softmax}(\mathbf{o})_j - y_j$
(that section), with no floor to cut it off.

We install the framework's loss as the method `loss` of `Classifier`, so that
every classification model in the book inherits it. The method takes
logits and integer labels. It first flattens all leading axes of both, since
later models predict one class for each position of a sequence, and returns
the mean loss, or with `averaged=False` the loss of every example.

In Keras the loss is `SparseCategoricalCrossentropy`. It assumes
probabilities unless we pass `from_logits=True`.

In [ ]:
@d2l.add_to_class(d2l.Classifier)
def loss(self, Y_hat, Y, averaged=True):
    Y_hat = d2l.reshape(Y_hat, (-1, Y_hat.shape[-1]))
    Y = d2l.reshape(Y, (-1,))
    reduction = (tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE
                 if averaged else tf.keras.losses.Reduction.NONE)
    fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction=reduction)
    return fn(Y, Y_hat)

The loss takes logits, not probabilities. A model whose `forward` ended
with a softmax would hand it probabilities, and the loss would apply a second
softmax to them. No error message appears, and the model still trains, but
its loss can no longer come near 0. This is why the `forward` of
`SoftmaxRegression` returns the logits.

We put the loss to the test on the logits $(1000, 0, 0)$ that broke the
softmax of the previous section, once with label 0 and once with label 1.
By the equation the losses are
$\log(1 + 2e^{-1000}) \approx 0$ and $1000 + \log(1 + 2e^{-1000}) \approx 1000$.

In [ ]:
O = d2l.tensor([[1000.0, 0.0, 0.0], [1000.0, 0.0, 0.0]])
y = d2l.tensor([0, 1])
SoftmaxRegression(num_outputs=3, lr=0.1).loss(O, y, averaged=False)

Both losses are finite and exact: the first example is classified correctly
with certainty, and the second is wrong by a margin of 1000.


## Training

We train with the same data, minibatch size, learning rate, and number of
epochs as in that section.

In [ ]:
data = d2l.FashionMNIST(batch_size=256)
model = SoftmaxRegression(num_outputs=10, lr=0.1)
trainer = d2l.Trainer(max_epochs=10)
trainer.fit(model, data)

After ten epochs the validation accuracy is about 83%, as for the
implementation from scratch, and the loss curves look alike. This is as it
should be. The framework's components change how the model is computed, not
the model. We now have a classifier, a loss that cannot blow up, and a
number, 83%. The rest of the chapter asks what the number is worth,
beginning with the probabilities behind it: can they be taken at face
value?

## Summary

Softmax regression with framework components is a fully connected layer on
flattened images whose forward pass returns logits, trained with the
framework's cross-entropy loss. The loss takes logits because computing it
from probabilities fails in floating point, by overflow of the exponential
or by the logarithm of a probability that has underflowed to 0. Written as
$l = \log \sum_k \exp(o_k) - o_y$ and evaluated after subtracting the
largest logit, the loss stays finite for any logits and keeps its gradient,
$\mathrm{softmax}(\mathbf{o}) - \mathbf{y}$. `Classifier.loss` installs this
loss for every classifier in the book. Trained for ten epochs, the model
reaches about 83% validation accuracy, like the version from scratch.

## Exercises

1. [code] **Double softmax.** The `loss` of this section expects logits.
   Suppose that a model's `forward` mistakenly ends with a softmax. The
   loss then receives a probability vector $\mathbf{p}$ in place of the
   logits and evaluates $-\log \mathrm{softmax}(\mathbf{p})_y$.
    1. Find the smallest value this loss can take over all probability
       vectors $\mathbf{p}$ and labels $y$, as a function of the number of
       classes $q$. Evaluate it for $q = 10$.
    1. Copy the from-scratch `softmax` of that section,
       which is not part of the `d2l` package. Subclass
       `SoftmaxRegression` so that `forward` applies this `softmax` to the
       logits. Train it with `lr=0.1`, `d2l.FashionMNIST(batch_size=256)`,
       and `d2l.Trainer(max_epochs=10)`. Compare its training loss and
       validation accuracy with those of `SoftmaxRegression`, and relate
       the final training loss to the value from the previous step.

1. [code] **Fused loss on extreme logits.** The text passes the logits
   $\mathbf{o} = (1000, 0, 0)$ to the `loss` of this section with labels 0
   and 1. Copy the from-scratch `softmax` of that section,
   which is not part of the `d2l` package.
    1. Compute the same two losses by the from-scratch route, `softmax`
       followed by `d2l.cross_entropy`. Explain each value it returns, and
       determine the largest finite loss this route can return.
    1. Verify on benign logits such as $\mathbf{o} = (2, 1, 0)$ that the
       two routes agree to floating-point precision.

1. [code] **Stable softplus.** For two classes with logits $(x, 0)$, the
   first term of the loss is $\log(1 + e^x)$, the *softplus* of $x$.
    1. Plot $\log(1 + e^x)$ and $\max(x, 0)$ on $[-5, 5]$. Show that the
       first exceeds the second by at most $\log 2$, at $x = 0$, and
       explain in one sentence why log-sum-exp is called a smooth maximum.
    1. Show that $\log(1 + e^x) = \max(x, 0) + \log(1 + e^{-|x|})$.
    1. In single precision, evaluate the naive expression
       `log(1 + exp(x))`, the rewritten expression, and the rewritten
       expression with `log1p` in place of `log(1 + ...)`, for
       $x \in \{-100, -20, -5, 0, 5, 20, 100\}$. Compare each value with a
       double-precision reference, and report where each version
       overflows and where it loses all relative accuracy.
    1. Compute the derivative of the naive expression at $x = -20$ by
       automatic differentiation. Explain why it is accurate although the
       value of the expression is not, and what this implies for training
       and for reporting the loss of confidently classified examples.

1. [code] **The gradient survives.** In that section,
   exercise 1, the floored loss of the logits $(0, -200, 0)$ with label 1
   had a zero gradient. Pass the same logits and label to the `loss` of
   this section and compute the gradient with respect to the logits by
   automatic differentiation. Compare it with $\hat{\mathbf{y}} - \mathbf{y}$
   from that section, and explain why this gradient would move the
   logit of class 1 while the floored one did not.

1. **Number formats.** Half-precision formats store a number in 16 bits:
   FP16 has 5 exponent bits and 10 mantissa bits, BFLOAT16 has 8 and 7, and
   FP32 has 8 and 23. Determine for each format the largest logit $o$ for
   which $\exp(o)$ does not overflow. Then explain why a network whose
   logits reach 20 can be trained in BFLOAT16 but not in FP16 unless its
   softmax is computed in FP32. Finally, consider the shifted sum of
   the equation for $q = 10^5$ classes, each term
   at most 1, accumulated in FP16: what goes wrong once the running sum
   exceeds 2,048, and what does the 10-bit mantissa have to do with it?

[Discussions](https://d2l.discourse.group/t/260)